# 📊 Data Lead Pipeline: Arabic + English Invoice Dataset (v1)
### Data Lead Responsibilities:
This notebook executes the full Data Lead delivery pipeline:
1. **Download & Convert** open-source datasets from Hugging Face (`CORD`, `SROIE`, `CORU`).
2. **Generate Synthetic Data** with ground truth for Arabic/English invoices (`generate_synthetic.py`).
3. **Validate Dataset** against the contract with 0 errors (`validate_dataset.py`).
4. **Create Splits & Check Leakage** with stratified Train / Val / Test (`create_splits.py`, `check_leakage.py`).
5. **Generate EDA Report** with statistical tables and charts (`run_eda.py`).
6. **Final Verification Check** to confirm dataset readiness.

## 1️⃣ Mount Google Drive and Setup Working Directory

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Navigate to project directory
import os

possible_paths = [
    '/content/drive/MyDrive/data-entry-ds',
    '/content/drive/MyDrive/project-dipe/data-entry-ds'
]

found = False
for p in possible_paths:
    if os.path.exists(p):
        os.chdir(p)
        print(f"✅ Successfully navigated to: {os.getcwd()}")
        found = True
        break

if not found:
    print("⚠️ Directory not found! Please make sure 'data-entry-ds' is uploaded to your Google Drive.")

## 2️⃣ Install Required Dependencies

In [ ]:
# Install dependencies for image processing, fonts, and Hugging Face datasets
!pip install -q datasets huggingface_hub arabic-reshaper python-bidi pillow matplotlib pandas tqdm opencv-python
!pip install -q playwright && playwright install --with-deps chromium

## 3️⃣ Download and Convert Datasets (Hugging Face & ICDAR)
This step automatically downloads and standardizes open-source receipt and invoice datasets into our JSONL data contract.

In [ ]:
# 1. Convert CORD v2 (Consolidated Receipt Dataset - 1,000 receipts)
!python scripts/convert_cord.py

# 2. Download and convert SROIE (ICDAR 2019 receipts)
!git clone https://huggingface.co/datasets/jsdnrs/ICDAR2019-SROIE data/raw/sroie
!python scripts/convert_sroie.py

## 4️⃣ Generate Synthetic Invoices (Arabic + English + Mixed)
Generates realistic synthetic invoices with complete ground truth (`invoice_number`, `date`, `supplier`, `total`).

In [ ]:
# Step A: Quick pilot generation (200 documents)
!python scripts/generate_synthetic.py --count 200 --pilot

# Step B: Full generation (target 8,000 documents with resume support)
!python scripts/generate_synthetic.py --count 8000 --resume

## 5️⃣ Validate Dataset (Quality & Schema Enforcement)
Validates all records to ensure 0 missing required fields and strict adherence to the data contract.

In [ ]:
!python scripts/validate_dataset.py

## 6️⃣ Create Train / Val / Test Splits & Check for Data Leakage
Performs stratified splitting and verifies that no document appears in more than one split.

In [ ]:
# 1. Create stratified splits
!python scripts/create_splits.py

# 2. Run data leakage verification
!python scripts/check_leakage.py

## 7️⃣ Generate Exploratory Data Analysis (EDA) Report
Generates charts and statistical summaries (saved to `docs/eda_report.md`).

In [ ]:
!python scripts/run_eda.py

## 8️⃣ Final Dataset Verification Check

In [ ]:
import json
from pathlib import Path

print("=" * 60)
print("🔍 FINAL DATASET READINESS VERIFICATION")
print("=" * 60)

splits = ["train", "val", "test"]
all_splits_ok = True
for s in splits:
    p = Path(f"data/splits/{s}.jsonl")
    if p.exists():
        count = sum(1 for _ in open(p, encoding="utf-8"))
        print(f"  ✅ Split {s.upper():<5} : {count:,} records")
    else:
        print(f"  ❌ Split {s.upper():<5} : NOT FOUND!")
        all_splits_ok = False

img_dir = Path("data/processed/images")
synth_img_dir = Path("data/synth/out/images")
total_images = (len(list(img_dir.glob("*.jpg"))) if img_dir.exists() else 0) + \
               (len(list(synth_img_dir.glob("*.jpg"))) if synth_img_dir.exists() else 0)
print(f"  ✅ Total Images  : {total_images:,} images found")

sample_path = Path("data/splits/train.jsonl")
if sample_path.exists():
    with open(sample_path, encoding="utf-8") as f:
        sample = json.loads(f.readline())
        print("\n📄 Sample Training Record:")
        print("  - Doc ID:         ", sample.get("doc_id"))
        print("  - Source:         ", sample.get("source"))
        print("  - Language:       ", sample.get("language"))
        print("  - Supplier:       ", sample.get("ground_truth", {}).get("supplier"))
        print("  - Invoice Number: ", sample.get("ground_truth", {}).get("invoice_number"))
        print("  - Date:           ", sample.get("ground_truth", {}).get("date"))
        print("  - Total:          ", sample.get("ground_truth", {}).get("total"))

print("=" * 60)
if all_splits_ok and total_images > 0:
    print("🎉 SUCCESS! Dataset v1 is completely ready and verified!")
else:
    print("⚠️ Please review missing steps above.")
print("=" * 60)